# Hospital Readmission Decision Support Dashboard

This notebook builds the interactive dashboard for the hospital readmission project.

The dashboard includes:

- 30-day readmission risk prediction using the final XGBoost model
- an adjustable intervention threshold
- evidence-based discharge support using RAG with AHRQ and CMS guidance
- interactive cost, value, and resource-impact analysis
- visualizations
- limitations and appropriate use guidance

### Files needed in the Colab session

Upload these files before running the notebook:

- `readmission_xgboost_model.pkl`
- `readmission_threshold.pkl`
- `rag_document_chunks.csv`
- `cleaned_diabetes_data.csv`

The OpenAI API key should be stored in Colab Secrets as `OPENAI_API_KEY`.


##  Install Libraries


In [ ]:
!pip install -q gradio joblib openai scikit-learn xgboost


## Imports


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from openai import OpenAI
from sklearn.neighbors import NearestNeighbors


## Upload the Final Model and Project Files


In [ ]:
# Helper function so the notebook still works if Colab added "(2)" to a file name
def first_existing_file(file_names):
    for file_name in file_names:
        if os.path.exists(file_name):
            return file_name

    raise FileNotFoundError(
        "Could not find any of these files: "
        + ", ".join(file_names)
    )


model_file = first_existing_file([
    "readmission_xgboost_model.pkl",
    "readmission_xgboost_model (2).pkl",
])

threshold_file = first_existing_file([
    "readmission_threshold.pkl",
    "readmission_threshold (2).pkl",
])

# Uploading the final reduced XGBoost pipeline
model = joblib.load(model_file)

# Uploading the recommended default threshold
default_threshold = joblib.load(threshold_file)

# Uploading the RAG document chunks
chunks_df = pd.read_csv(
    "rag_document_chunks.csv"
)

# Uploading the cleaned data to create valid dashboard dropdown choices
data = pd.read_csv(
    "cleaned_diabetes_data.csv"
)

print("Model loaded successfully.")
print("Default threshold:", default_threshold)
print("Expected model inputs:")
print(model.feature_names_in_.tolist())

print("\nRAG chunks loaded:", len(chunks_df))
print("Cleaned dataset shape:", data.shape)


Model loaded successfully.
Default threshold: 0.4
Expected model inputs:
['age', 'admission_type_id', 'discharge_disposition_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'A1Cresult', 'diabetesMed', 'diag_1_group', 'diag_2_group', 'diag_3_group']

RAG chunks loaded: 85
Cleaned dataset shape: (99235, 44)


## Connect OpenAI


In [ ]:
from google.colab import userdata

openai_api_key = userdata.get(
    "OPENAI_API_KEY"
)

if not openai_api_key:
    raise RuntimeError(
        "OPENAI_API_KEY was not found in Colab Secrets."
    )

client = OpenAI(
    api_key=openai_api_key
)

print("OpenAI client initialized.")


OpenAI client initialized.


## Create Dashboard Input Options


In [ ]:
# Getting the available values for the categorical inputs
age_options = sorted(
    data["age"].dropna().unique().tolist()
)

admission_type_options = sorted(
    data["admission_type_id"].dropna().unique().tolist()
)

discharge_options = sorted(
    data["discharge_disposition_id"].dropna().unique().tolist()
)

a1c_options = sorted(
    data["A1Cresult"].dropna().unique().tolist()
)

diabetes_med_options = sorted(
    data["diabetesMed"].dropna().unique().tolist()
)

diag_1_options = sorted(
    data["diag_1_group"].dropna().unique().tolist()
)

diag_2_options = sorted(
    data["diag_2_group"].dropna().unique().tolist()
)

diag_3_options = sorted(
    data["diag_3_group"].dropna().unique().tolist()
)

print("Dashboard input options created.")


Dashboard input options created.


## Readmission Risk Functions


In [ ]:
def make_risk_plot(probability, threshold):
    # Showing the patient's estimated risk compared with the selected threshold
    fig, ax = plt.subplots(figsize=(7, 2.2))

    ax.barh(
        ["Patient"],
        [probability * 100],
        color="#4f86a6",
        height=0.45
    )

    ax.axvline(
        threshold * 100,
        color="#b44b4b",
        linestyle="--",
        linewidth=2,
        label=f"Threshold ({threshold:.2f})"
    )

    ax.set_xlim(0, 100)
    ax.set_xlabel("Estimated 30-Day Readmission Risk (%)")
    ax.set_title("Patient Risk Compared With Selected Threshold")
    ax.legend(loc="lower right")
    ax.grid(axis="x", alpha=0.15)

    plt.tight_layout()

    return fig


def create_risk_interpretation(probability, selected_threshold):
    prediction = int(
        probability >= selected_threshold
    )

    if prediction == 1:
        risk_label = "Additional Support Recommended"

        interpretation = (
            f"This patient's predicted readmission risk is "
            f"{probability * 100:.1f}%, which is above the "
            f"{selected_threshold:.2f} threshold. Additional "
            f"post-discharge support should be considered."
        )

    else:
        risk_label = "Standard Discharge Planning"

        interpretation = (
            f"This patient's predicted readmission risk is "
            f"{probability * 100:.1f}%, which is below the "
            f"{selected_threshold:.2f} threshold. Standard "
            f"discharge planning may be appropriate, while "
            f"clinical judgment should still be used."
        )

    return risk_label, interpretation


def predict_readmission(
    age,
    admission_type_id,
    discharge_disposition_id,
    time_in_hospital,
    num_lab_procedures,
    num_procedures,
    num_medications,
    number_outpatient,
    number_emergency,
    number_inpatient,
    number_diagnoses,
    A1Cresult,
    diabetesMed,
    diag_1_group,
    diag_2_group,
    diag_3_group,
    selected_threshold
):
    # Creating a dataframe with the patient information
    patient_df = pd.DataFrame({
        "age": [age],
        "admission_type_id": [admission_type_id],
        "discharge_disposition_id": [discharge_disposition_id],
        "time_in_hospital": [time_in_hospital],
        "num_lab_procedures": [num_lab_procedures],
        "num_procedures": [num_procedures],
        "num_medications": [num_medications],
        "number_outpatient": [number_outpatient],
        "number_emergency": [number_emergency],
        "number_inpatient": [number_inpatient],
        "number_diagnoses": [number_diagnoses],
        "A1Cresult": [A1Cresult],
        "diabetesMed": [diabetesMed],
        "diag_1_group": [diag_1_group],
        "diag_2_group": [diag_2_group],
        "diag_3_group": [diag_3_group],
    })

    # Getting the predicted probability of 30-day readmission
    probability = model.predict_proba(
        patient_df
    )[0, 1]

    risk_label, interpretation = create_risk_interpretation(
        probability,
        selected_threshold
    )

    risk_plot = make_risk_plot(
        probability,
        selected_threshold
    )

    return (
        f"{probability * 100:.1f}%",
        risk_label,
        interpretation,
        probability,
        risk_plot
    )


def update_threshold_result(
    raw_probability,
    selected_threshold
):
    # The model probability does not change when the threshold changes.
    # Only the support recommendation changes.
    if raw_probability is None:
        return (
            "Run a patient prediction first.",
            "Run a patient prediction first.",
            None
        )

    risk_label, interpretation = create_risk_interpretation(
        raw_probability,
        selected_threshold
    )

    risk_plot = make_risk_plot(
        raw_probability,
        selected_threshold
    )

    return (
        risk_label,
        interpretation,
        risk_plot
    )


## Build the RAG Retrieval System


In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"
RAG_MODEL = "gpt-5-nano"


def embed_texts(
    texts,
    model_name,
    batch_size=250
):
    texts = [
        str(text)
        for text in texts
    ]

    all_embeddings = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):
        batch = texts[
            start:start + batch_size
        ]

        response = client.embeddings.create(
            model=model_name,
            input=batch
        )

        all_embeddings.extend(
            item.embedding
            for item in response.data
        )

    return np.asarray(
        all_embeddings,
        dtype=np.float32
    )


# Creating embeddings for the saved RAG document chunks
chunk_embeddings = embed_texts(
    chunks_df["text"].tolist(),
    EMBEDDING_MODEL
)

# Building the document retrieval index
document_index = NearestNeighbors(
    n_neighbors=5,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

document_index.fit(
    chunk_embeddings
)

print("RAG vector store ready.")


RAG vector store ready.


In [ ]:
def retrieve_guidance(
    query,
    top_k=5
):
    query_embedding = embed_texts(
        [query],
        EMBEDDING_MODEL
    )

    distances, positions = (
        document_index.kneighbors(
            query_embedding,
            n_neighbors=top_k
        )
    )

    similarities = (
        1.0 - distances[0]
    )

    results = chunks_df.iloc[
        positions[0]
    ].copy()

    results["similarity"] = similarities

    results["retrieval_rank"] = range(
        1,
        len(results) + 1
    )

    return results


RAG_SYSTEM_PROMPT = """
You are a hospital readmission decision-support assistant.

Use only the retrieved AHRQ and CMS guidance provided in the prompt.

Provide concise, practical post-discharge recommendations for patients
who may be at risk of 30-day hospital readmission.

Requirements:
- Base every recommendation only on the retrieved evidence.
- Do not add recommendations that are not supported by the guidance.
- Include the source title and page number for every recommendation.
- Focus on discharge planning, follow-up, medication management,
  patient education, and care coordination.
- Do not diagnose the patient or replace clinical judgment.
- If the retrieved evidence does not support a recommendation,
  do not include it.
- Do not offer additional help or ask follow-up questions.
- End after the recommendations and source references.
""".strip()


def build_rag_prompt(
    query,
    retrieved_chunks
):
    evidence = []

    for _, row in retrieved_chunks.iterrows():
        evidence.append(
            f"Source: {row['title']} | "
            f"Page: {row['page']}\n"
            f"{row['text']}"
        )

    evidence_text = "\n\n".join(
        evidence
    )

    return f"""
Question:
{query}

Retrieved Guidance:
{evidence_text}

Using only the retrieved guidance, provide 3 practical
post-discharge recommendations.

For each recommendation:
1. State the recommended action.
2. Give one brief sentence explaining why it may help.
3. Cite the source title and page number.

Do not include information that is not supported by the retrieved evidence.
Prioritize the three recommendations that are most directly relevant
to the user's question. Avoid repetitive or overlapping recommendations.
Keep each recommendation concise.

"""


def generate_rag_answer(query):
    if not query or not query.strip():
        return (
            "Enter a question about discharge planning, "
            "follow-up, medications, or readmission support."
        )

    retrieved = retrieve_guidance(
        query,
        top_k=5
    )

    prompt = build_rag_prompt(
        query,
        retrieved
    )

    response = client.responses.create(
        model=RAG_MODEL,
        instructions=RAG_SYSTEM_PROMPT,
        input=prompt,
        reasoning={
            "effort": "minimal"
        },
        max_output_tokens=800,
        store=False,
    )

    return response.output_text.strip()


## Cost, Value, and Resource-Impact Functions




In [ ]:
# Final XGBoost test-set results used for resource-impact scenarios
xgb_threshold_reference = pd.DataFrame({
    "Threshold": [
        0.20,
        0.30,
        0.40,
        0.50,
        0.60
    ],
    "Patients Flagged": [
        19626,
        17555,
        12006,
        6891,
        2919
    ],
    "Readmissions Caught": [
        2253,
        2162,
        1779,
        1279,
        718
    ],
    "Readmissions Missed": [
        7,
        98,
        481,
        981,
        1542
    ]
})

display(xgb_threshold_reference)


,Threshold,Patients Flagged,Readmissions Caught,Readmissions Missed
0,0.2,19626,2253,7
1,0.3,17555,2162,98
2,0.4,12006,1779,481
3,0.5,6891,1279,981
4,0.6,2919,718,1542


In [ ]:
def make_cost_plot(
    estimated_savings,
    intervention_cost,
    net_value
):
    labels = [
        "Estimated Savings",
        "Intervention Cost",
        "Net Value"
    ]

    values = [
        estimated_savings,
        intervention_cost,
        net_value
    ]

    fig, ax = plt.subplots(
        figsize=(7, 4)
    )

    ax.bar(
        labels,
        values,
        color=[
            "#4f86a6",
            "#8ca6b5",
            "#2f6f8f"
        ]
    )

    ax.set_ylabel("Estimated Dollars")
    ax.set_title("Estimated Financial Impact")
    ax.ticklabel_format(
        style="plain",
        axis="y"
    )
    ax.grid(
        axis="y",
        alpha=0.15
    )

    plt.xticks(
        rotation=10
    )

    plt.tight_layout()

    return fig


def calculate_cost_impact(
    threshold,
    cost_per_readmission,
    cost_per_intervention,
    intervention_effectiveness_percent
):
    # Matching the selected threshold to the saved test-set results
    selected_row = xgb_threshold_reference[
        np.isclose(
            xgb_threshold_reference["Threshold"],
            float(threshold)
        )
    ]

    if selected_row.empty:
        raise ValueError(
            "Choose one of the tested thresholds: "
            "0.20, 0.30, 0.40, 0.50, or 0.60."
        )

    selected_row = selected_row.iloc[0]

    patients_flagged = int(
        selected_row["Patients Flagged"]
    )

    readmissions_caught = int(
        selected_row["Readmissions Caught"]
    )

    readmissions_missed = int(
        selected_row["Readmissions Missed"]
    )

    intervention_effectiveness = (
        intervention_effectiveness_percent
        / 100
    )

    # Estimating avoided readmissions
    expected_avoided = (
        readmissions_caught
        * intervention_effectiveness
    )

    # Estimating financial impact
    estimated_savings = (
        expected_avoided
        * cost_per_readmission
    )

    intervention_cost = (
        patients_flagged
        * cost_per_intervention
    )

    net_value = (
        estimated_savings
        - intervention_cost
    )

    if intervention_cost > 0:
        roi = (
            net_value
            / intervention_cost
        ) * 100
    else:
        roi = 0

    if readmissions_caught > 0:
        break_even = (
            intervention_cost
            /
            (
                readmissions_caught
                * cost_per_readmission
            )
        ) * 100
    else:
        break_even = 0

    cost_plot = make_cost_plot(
        estimated_savings,
        intervention_cost,
        net_value
    )

    interpretation = (
        f"At a {float(threshold):.2f} threshold, the model flags "
        f"{patients_flagged:,} patients and identifies "
        f"{readmissions_caught:,} readmissions in the test data. "
        f"Under the selected assumptions, the estimated net value is "
        f"{net_value:,.0f} dollars. Lower thresholds generally identify "
        f"more readmissions but require more intervention resources."
    )

    return (
        f"{patients_flagged:,}",
        f"{readmissions_caught:,}",
        f"{readmissions_missed:,}",
        f"{expected_avoided:,.0f}",
        f"{intervention_cost:,.0f} dollars",
        f"{estimated_savings:,.0f} dollars",
        f"{net_value:,.0f} dollars",
        f"{roi:.1f}%",
        f"{break_even:.1f}%",
        interpretation,
        cost_plot
    )


In [ ]:
# Diagnosis options for the dashboard
diagnosis_options = [
    "Circulatory",
    "Diabetes",
    "Digestive",
    "Genitourinary",
    "Injury",
    "Musculoskeletal",
    "Neoplasms",
    "Other",
    "Respiratory",
    "Unknown"
]

## Build the Dashboard


In [ ]:
# Creating a hospital-style theme
hospital_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="blue",
    neutral_hue="slate"
)

custom_css = """
.gradio-container {
    max-width: 1260px !important;
    margin: auto !important;
    background: #f7fafc;
}

.hospital-header {
    background: linear-gradient(135deg, #e7f2f8, #ffffff);
    border: 1px solid #d4e3ec;
    border-radius: 18px;
    padding: 28px 32px;
    margin-bottom: 20px;
}

.threshold-box,
.info-card,
.guidance-card {
    background: #ffffff;
    border: 1px solid #d8e5ed;
    border-radius: 14px;
    padding: 16px;
    margin-bottom: 14px;
}

.threshold-box {
    background: #eef6fa;
    border-left: 4px solid #4f86a6;
}

button.primary {
    background: #2f6f8f !important;
    border-color: #2f6f8f !important;
}

button.primary:hover {
    background: #255c77 !important;
    border-color: #255c77 !important;
}

.tabs button.selected {
    color: #245f7d !important;
    border-color: #4f86a6 !important;
}

footer {
    display: none !important;
}
"""


# Building the dashboard
with gr.Blocks(
    theme=hospital_theme,
    css=custom_css,
    title="Hospital Readmission Decision Support"
) as demo:

    # ---------------------------------------------------------
    # DASHBOARD HEADER
    # ---------------------------------------------------------

    gr.HTML(
        """
        <div class="hospital-header">

            <div style="font-size: 36px; margin-bottom: 4px;">
                🏥
            </div>

            <h1 style="margin-bottom: 8px;">
                Hospital Readmission Decision Support
            </h1>

            <p style="font-size: 17px; margin-bottom: 5px;">
                Enter patient information to estimate 30-day readmission
                risk and identify patients who may benefit from extra
                support after discharge.
            </p>

            <p style="color: #526777; margin-bottom: 0;">
                Use the tabs below to explore patient risk,
                discharge guidance, and estimated hospital impact.
            </p>

        </div>
        """
    )


    # =========================================================
    # TAB 1: READMISSION RISK
    # =========================================================

    with gr.Tab("Readmission Risk"):

        gr.Markdown(
            """
            ## Patient Information

            Enter the patient's information below.
            """
        )

        # Patient and encounter information
        with gr.Row():

            age = gr.Dropdown(
                choices=age_options,
                value="[70-80)",
                label="Age"
            )

            admission_type_id = gr.Dropdown(
                choices=admission_type_options,
                value=1,
                label="Admission Type"
            )

            discharge_disposition_id = gr.Dropdown(
                choices=discharge_options,
                value=1,
                label="Discharge Disposition"
            )

            time_in_hospital = gr.Number(
                value=5,
                label="Days in Hospital"
            )


        # Hospital stay information
        with gr.Row():

            num_lab_procedures = gr.Number(
                value=40,
                label="Lab Procedures"
            )

            num_procedures = gr.Number(
                value=1,
                label="Procedures"
            )

            num_medications = gr.Number(
                value=15,
                label="Medications"
            )

            number_diagnoses = gr.Number(
                value=7,
                label="Diagnoses"
            )


        # Previous healthcare use
        gr.Markdown("### Previous Healthcare Use")

        with gr.Row():

            number_inpatient = gr.Number(
                value=0,
                label="Previous Inpatient Visits"
            )

            number_emergency = gr.Number(
                value=0,
                label="Previous Emergency Visits"
            )

            number_outpatient = gr.Number(
                value=0,
                label="Previous Outpatient Visits"
            )


        # Diabetes information
        gr.Markdown("### Diabetes and Diagnoses")

        with gr.Row():

            a1c_result = gr.Dropdown(
                choices=a1c_options,
                value="Not Recorded",
                label="A1C Result"
            )

            diabetes_med = gr.Dropdown(
                choices=diabetes_med_options,
                value="Yes",
                label="Taking Diabetes Medication"
            )


        with gr.Row():

            diag_1_group = gr.Dropdown(
                choices=diagnosis_options,
                value="Circulatory",
                label="Primary Diagnosis"
            )

            diag_2_group = gr.Dropdown(
                choices=diagnosis_options,
                value="Diabetes",
                label="Secondary Diagnosis"
            )

            diag_3_group = gr.Dropdown(
                choices=diagnosis_options,
                value="Other",
                label="Additional Diagnosis"
            )


        # -----------------------------------------------------
        # RISK SETTINGS
        # -----------------------------------------------------

        gr.Markdown("## Risk Settings")

        with gr.Row():

            selected_threshold = gr.Slider(
                minimum=0.20,
                maximum=0.60,
                value=default_threshold,
                step=0.05,
                label="Readmission Threshold",
                scale=3
            )

            predict_button = gr.Button(
                "Check Readmission Risk",
                variant="primary",
                scale=1
            )


        gr.Markdown(
            """
            **Recommended starting threshold: 0.40.**
            Lower thresholds identify more patients for support,
            while higher thresholds flag fewer patients.
            """
        )


        # -----------------------------------------------------
        # RESULTS
        # -----------------------------------------------------

        gr.Markdown("## Results")

        with gr.Row():

            risk_probability = gr.Textbox(
                label="Estimated 30-Day Readmission Risk"
            )

            risk_classification = gr.Textbox(
                label="Support Recommendation"
            )


        # Interpretation and visualization side-by-side
        with gr.Row():

            with gr.Column(scale=1):

                risk_interpretation = gr.Textbox(
                    label="What This Means",
                    lines=6
                )

            with gr.Column(scale=1):

                risk_plot = gr.Plot(
                    label=False
                )


        raw_probability = gr.Number(
            visible=False
        )


        # Running the prediction
        predict_button.click(
            fn=predict_readmission,
            inputs=[
                age,
                admission_type_id,
                discharge_disposition_id,
                time_in_hospital,
                num_lab_procedures,
                num_procedures,
                num_medications,
                number_outpatient,
                number_emergency,
                number_inpatient,
                number_diagnoses,
                a1c_result,
                diabetes_med,
                diag_1_group,
                diag_2_group,
                diag_3_group,
                selected_threshold
            ],
            outputs=[
                risk_probability,
                risk_classification,
                risk_interpretation,
                raw_probability,
                risk_plot
            ]
        )


        # Updating results when the threshold changes
        selected_threshold.change(
            fn=update_threshold_result,
            inputs=[
                raw_probability,
                selected_threshold
            ],
            outputs=[
                risk_classification,
                risk_interpretation,
                risk_plot
            ]
        )


    # =========================================================
    # TAB 2: DISCHARGE SUPPORT
    # =========================================================

    with gr.Tab("Discharge Support"):

        gr.Markdown(
            """
            ## Evidence-Based Discharge Support

            Use this section to explore ways to support patients
            after discharge and help reduce their risk of readmission.

            Ask a question about **follow-up care, medications,
            patient education, discharge planning, or care coordination**.
            The tool searches the available **AHRQ and CMS guidance**
            and uses the most relevant information to provide practical
            recommendations.

            Recommendations are meant to support discharge planning
            and should be considered alongside clinical judgment.
            """
        )


        rag_question = gr.Textbox(
            label="Ask a Question",
            placeholder=(
                "Example: What follow-up support should be provided "
                "for a patient at high risk of readmission?"
            ),
            lines=3
        )


        rag_button = gr.Button(
            "Get Discharge Guidance",
            variant="primary"
        )


        gr.Markdown("### Recommendations")

        rag_answer = gr.Markdown()


        rag_button.click(
            fn=generate_rag_answer,
            inputs=rag_question,
            outputs=rag_answer
        )


    # =========================================================
    # TAB 3: RESOURCE & VALUE IMPACT
    # =========================================================

    with gr.Tab("Resource & Value Impact"):

        gr.Markdown(
            """
            ## Resource & Value Impact

            Adjust the settings below to see how different thresholds
            and intervention assumptions could affect hospital resources
            and estimated value.
            """
        )


        # -----------------------------------------------------
        # SCENARIO SETTINGS
        # -----------------------------------------------------

        gr.Markdown("### Scenario Settings")

        with gr.Row():

            cost_threshold = gr.Dropdown(
                choices=[
                    0.20,
                    0.30,
                    0.40,
                    0.50,
                    0.60
                ],
                value=0.40,
                label="Readmission Threshold"
            )

            cost_per_readmission_input = gr.Number(
                value=16300,
                label="Estimated Cost per Readmission"
            )

            cost_per_intervention_input = gr.Number(
                value=200,
                label="Cost per Patient Intervention"
            )

            intervention_effectiveness_input = gr.Slider(
                minimum=5,
                maximum=50,
                value=20,
                step=5,
                label="Expected Intervention Effectiveness (%)"
            )


        cost_button = gr.Button(
            "Calculate Resource Impact",
            variant="primary"
        )


        gr.Markdown(
            """
            Change these values to explore different hospital scenarios.
            """
        )


        # -----------------------------------------------------
        # MAIN RESULTS
        # -----------------------------------------------------

        gr.Markdown("## Estimated Impact")

        with gr.Row():

            patients_flagged_output = gr.Textbox(
                label="Patients Flagged"
            )

            readmissions_caught_output = gr.Textbox(
                label="Readmissions Identified"
            )

            net_value_output = gr.Textbox(
                label="Estimated Net Value"
            )

            roi_output = gr.Textbox(
                label="Estimated ROI"
            )


        # -----------------------------------------------------
        # ADDITIONAL DETAILS
        # -----------------------------------------------------

        gr.Markdown("### Additional Details")

        with gr.Row():

            readmissions_missed_output = gr.Textbox(
                label="Readmissions Missed"
            )

            avoided_output = gr.Textbox(
                label="Expected Readmissions Avoided"
            )

            intervention_cost_output = gr.Textbox(
                label="Intervention Cost"
            )


        with gr.Row():

            savings_output = gr.Textbox(
                label="Estimated Savings"
            )

            break_even_output = gr.Textbox(
                label="Break-Even Effectiveness"
            )


        # -----------------------------------------------------
        # INTERPRETATION AND VISUALIZATION
        # -----------------------------------------------------

        with gr.Row():

            with gr.Column(scale=1):

                gr.Markdown("### What This Means")

                cost_interpretation_output = gr.Textbox(
                    label=False,
                    lines=6
                )

            with gr.Column(scale=1):

                gr.Markdown("### Financial Impact")

                cost_plot_output = gr.Plot(
                    label=False
                )


        cost_button.click(
            fn=calculate_cost_impact,
            inputs=[
                cost_threshold,
                cost_per_readmission_input,
                cost_per_intervention_input,
                intervention_effectiveness_input
            ],
            outputs=[
                patients_flagged_output,
                readmissions_caught_output,
                readmissions_missed_output,
                avoided_output,
                intervention_cost_output,
                savings_output,
                net_value_output,
                roi_output,
                break_even_output,
                cost_interpretation_output,
                cost_plot_output
            ]
        )


    # =========================================================
    # TAB 4: ABOUT & LIMITATIONS
    # =========================================================

    with gr.Tab("About & Limitations"):

        gr.Markdown(
            """
            ## About This Tool

            This dashboard was created to help identify patients
            who may be at higher risk of 30-day readmission and
            could benefit from additional support after discharge.

            It combines an XGBoost prediction model, evidence-based
            discharge guidance, and a hospital cost/value analysis.

            ## Important to Know

            This tool is meant to support decision-making,
            not replace clinical judgment.

            - The model was trained on historical data from patients with diabetes.
            - Performance may differ with other hospitals or patient populations.
            - The model can still miss readmissions or flag patients who are not readmitted.
            - Cost and savings estimates depend on the assumptions entered.
            - Discharge recommendations are based on a limited set of AHRQ and CMS guidance.
            """
        )


/tmp/ipykernel_4099/1177799779.py:60: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


## 10. Launch the Dashboard


In [ ]:
demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c1c20cfba3172faff.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from pathlib import Path

app_path = Path(
    "/content/hospital_readmission_space/app.py"
)

# Reading the current deployed app
app_text = app_path.read_text(
    encoding="utf-8"
)


app_text = app_text.replace(
    'fig, ax = plt.subplots(figsize=(7, 2.2))',
    'fig, ax = plt.subplots(figsize=(6.5, 2.4))'
)

app_text = app_text.replace(
    'ax.set_title("Patient Risk Compared With Selected Threshold")',
    'ax.set_title("Readmission Risk vs. Threshold", pad=12)'
)

app_text = app_text.replace(
    'figsize=(7, 4)',
    'figsize=(6.5, 4)'
)

app_text = app_text.replace(
    'ax.set_title("Estimated Financial Impact")',
    'ax.set_title("Financial Impact", pad=12)'
)

app_text = app_text.replace(
    'f"{intervention_cost:,.0f} dollars"',
    'f"${intervention_cost:,.0f}"'
)

app_text = app_text.replace(
    'f"{estimated_savings:,.0f} dollars"',
    'f"${estimated_savings:,.0f}"'
)

app_text = app_text.replace(
    'f"{net_value:,.0f} dollars"',
    'f"${net_value:,.0f}"'
)

app_text = app_text.replace(
    'f"{net_value:,.0f} dollars. Lower thresholds generally identify "',
    'f"${net_value:,.0f}. Lower thresholds generally identify "'
)




start_marker = (
    "# ---------------------------------------------------------\n"
    "# BUILD DASHBOARD\n"
    "# ---------------------------------------------------------"
)

end_marker = (
    '\n\nif __name__ == "__main__":'
)

start = app_text.index(
    start_marker
)

end = app_text.index(
    end_marker
)


new_dashboard = r'''

with gr.Blocks(
    theme=hospital_theme,
    css=custom_css,
    title="Hospital Readmission Decision Support"
) as demo:

    gr.HTML(
        """
        <div class="hospital-header">

            <div style="font-size: 36px; margin-bottom: 4px;">
                🏥
            </div>

            <h1 style="margin-bottom: 8px;">
                Hospital Readmission Decision Support
            </h1>

            <p style="font-size: 17px; margin-bottom: 5px;">
                Enter patient information to estimate 30-day readmission
                risk and identify patients who may benefit from extra
                support after discharge.
            </p>

            <p style="color: #526777; margin-bottom: 0;">
                Use the tabs below to explore patient risk,
                discharge guidance, and estimated hospital impact.
            </p>

        </div>
        """
    )



    with gr.Tab("Readmission Risk"):

        gr.Markdown(
            """
            ## Patient Information

            Enter the patient's information below.
            """
        )

        with gr.Row():

            age = gr.Dropdown(
                choices=age_options,
                value="[70-80)",
                label="Age"
            )

            admission_type_id = gr.Dropdown(
                choices=admission_type_options,
                value=1,
                label="Admission Type"
            )

            discharge_disposition_id = gr.Dropdown(
                choices=discharge_options,
                value=1,
                label="Discharge Disposition"
            )

            time_in_hospital = gr.Number(
                value=5,
                label="Days in Hospital"
            )


        with gr.Row():

            num_lab_procedures = gr.Number(
                value=40,
                label="Lab Procedures"
            )

            num_procedures = gr.Number(
                value=1,
                label="Procedures"
            )

            num_medications = gr.Number(
                value=15,
                label="Medications"
            )

            number_diagnoses = gr.Number(
                value=7,
                label="Diagnoses"
            )


        gr.Markdown(
            "### Previous Healthcare Use"
        )

        with gr.Row():

            number_inpatient = gr.Number(
                value=0,
                label="Previous Inpatient Visits"
            )

            number_emergency = gr.Number(
                value=0,
                label="Previous Emergency Visits"
            )

            number_outpatient = gr.Number(
                value=0,
                label="Previous Outpatient Visits"
            )


        gr.Markdown(
            "### Diabetes and Diagnoses"
        )

        with gr.Row():

            a1c_result = gr.Dropdown(
                choices=a1c_options,
                value="Not Recorded",
                label="A1C Result"
            )

            diabetes_med = gr.Dropdown(
                choices=diabetes_med_options,
                value="Yes",
                label="Taking Diabetes Medication"
            )


        with gr.Row():

            diag_1_group = gr.Dropdown(
                choices=diagnosis_options,
                value="Circulatory",
                label="Primary Diagnosis"
            )

            diag_2_group = gr.Dropdown(
                choices=diagnosis_options,
                value="Diabetes",
                label="Secondary Diagnosis"
            )

            diag_3_group = gr.Dropdown(
                choices=diagnosis_options,
                value="Other",
                label="Additional Diagnosis"
            )


        gr.Markdown(
            "## Risk Settings"
        )

        with gr.Row():

            selected_threshold = gr.Slider(
                minimum=0.20,
                maximum=0.60,
                value=default_threshold,
                step=0.05,
                label="Readmission Threshold",
                scale=3
            )

            predict_button = gr.Button(
                "Check Readmission Risk",
                variant="primary",
                scale=1
            )


        gr.Markdown(
            """
            **Recommended starting threshold: 0.40.**
            Lower thresholds identify more patients for support,
            while higher thresholds flag fewer patients.
            """
        )


        gr.Markdown(
            "## Results"
        )

        with gr.Row():

            risk_probability = gr.Textbox(
                label="Estimated 30-Day Readmission Risk"
            )

            risk_classification = gr.Textbox(
                label="Support Recommendation"
            )


        with gr.Row():

            with gr.Column(scale=1):

                risk_interpretation = gr.Textbox(
                    label="What This Means",
                    lines=6
                )

            with gr.Column(scale=1):

                risk_plot = gr.Plot(
                    label=None
                )


        raw_probability = gr.Number(
            visible=False
        )


        predict_button.click(
            fn=predict_readmission,
            inputs=[
                age,
                admission_type_id,
                discharge_disposition_id,
                time_in_hospital,
                num_lab_procedures,
                num_procedures,
                num_medications,
                number_outpatient,
                number_emergency,
                number_inpatient,
                number_diagnoses,
                a1c_result,
                diabetes_med,
                diag_1_group,
                diag_2_group,
                diag_3_group,
                selected_threshold
            ],
            outputs=[
                risk_probability,
                risk_classification,
                risk_interpretation,
                raw_probability,
                risk_plot
            ]
        )


        selected_threshold.change(
            fn=update_threshold_result,
            inputs=[
                raw_probability,
                selected_threshold
            ],
            outputs=[
                risk_classification,
                risk_interpretation,
                risk_plot
            ]
        )



    with gr.Tab("Discharge Support"):

        gr.Markdown(
            """
            ## Evidence-Based Discharge Support

            Use this section to explore ways to support patients
            after discharge and help reduce their risk of readmission.

            Ask a question about **follow-up care, medications,
            patient education, discharge planning, or care coordination**.
            The tool searches the available **AHRQ and CMS guidance**
            and uses the most relevant information to provide practical
            recommendations.

            Recommendations are meant to support discharge planning
            and should be considered alongside clinical judgment.
            """
        )


        rag_question = gr.Textbox(
            label="Ask a Question",
            placeholder=(
                "Example: What follow-up support should be provided "
                "for a patient at high risk of readmission?"
            ),
            lines=3
        )


        rag_button = gr.Button(
            "Get Discharge Guidance",
            variant="primary"
        )


        gr.Markdown(
            "### Recommendations"
        )

        rag_answer = gr.Markdown()


        rag_button.click(
            fn=generate_rag_answer,
            inputs=rag_question,
            outputs=rag_answer
        )



    with gr.Tab("Resource & Value Impact"):

        gr.Markdown(
            """
            ## Resource & Value Impact

            Adjust the settings below to see how different thresholds
            and intervention assumptions could affect hospital resources
            and estimated value.
            """
        )


        gr.Markdown(
            "### Scenario Settings"
        )

        with gr.Row():

            cost_threshold = gr.Dropdown(
                choices=[
                    0.20,
                    0.30,
                    0.40,
                    0.50,
                    0.60
                ],
                value=0.40,
                label="Readmission Threshold"
            )

            cost_per_readmission_input = gr.Number(
                value=16300,
                label="Estimated Cost per Readmission"
            )

            cost_per_intervention_input = gr.Number(
                value=200,
                label="Cost per Patient Intervention"
            )

            intervention_effectiveness_input = gr.Slider(
                minimum=5,
                maximum=50,
                value=20,
                step=5,
                label="Expected Intervention Effectiveness (%)"
            )


        cost_button = gr.Button(
            "Calculate Resource Impact",
            variant="primary"
        )


        gr.Markdown(
            """
            Change these values to explore different hospital scenarios.
            """
        )


        gr.Markdown(
            "## Estimated Impact"
        )

        with gr.Row():

            patients_flagged_output = gr.Textbox(
                label="Patients Flagged"
            )

            readmissions_caught_output = gr.Textbox(
                label="Readmissions Identified"
            )

            net_value_output = gr.Textbox(
                label="Estimated Net Value"
            )

            roi_output = gr.Textbox(
                label="Estimated ROI"
            )


        gr.Markdown(
            "### Additional Details"
        )

        with gr.Row():

            readmissions_missed_output = gr.Textbox(
                label="Readmissions Missed"
            )

            avoided_output = gr.Textbox(
                label="Expected Readmissions Avoided"
            )

            intervention_cost_output = gr.Textbox(
                label="Intervention Cost"
            )


        with gr.Row():

            savings_output = gr.Textbox(
                label="Estimated Savings"
            )

            break_even_output = gr.Textbox(
                label="Break-Even Effectiveness"
            )


        with gr.Row():

            with gr.Column(scale=1):

                gr.Markdown(
                    "### What This Means"
                )

                cost_interpretation_output = gr.Textbox(
                    label=None,
                    lines=6
                )

            with gr.Column(scale=1):

                gr.Markdown(
                    "### Financial Impact"
                )

                cost_plot_output = gr.Plot(
                    label=None
                )


        cost_button.click(
            fn=calculate_cost_impact,
            inputs=[
                cost_threshold,
                cost_per_readmission_input,
                cost_per_intervention_input,
                intervention_effectiveness_input
            ],
            outputs=[
                patients_flagged_output,
                readmissions_caught_output,
                readmissions_missed_output,
                avoided_output,
                intervention_cost_output,
                savings_output,
                net_value_output,
                roi_output,
                break_even_output,
                cost_interpretation_output,
                cost_plot_output
            ]
        )


    with gr.Tab("About & Limitations"):

        gr.Markdown(
            """
            ## About This Tool

            This dashboard was created to help identify patients
            who may be at higher risk of 30-day readmission and
            could benefit from additional support after discharge.

            It combines an XGBoost prediction model,
            evidence-based discharge guidance, and a hospital
            cost/value analysis.

            ## Important to Know

            This tool is meant to support decision-making,
            not replace clinical judgment.

            - The model was trained on historical data from patients with diabetes.
            - Performance may differ with other hospitals or patient populations.
            - The model can still miss readmissions or flag patients who are not readmitted.
            - Cost and savings estimates depend on the assumptions entered.
            - Discharge recommendations are based on a limited set of AHRQ and CMS guidance.
            """
        )
'''


updated_app = (
    app_text[:start]
    + new_dashboard
    + app_text[end:]
)

# Saving directly over the deployment app.py
app_path.write_text(
    updated_app,
    encoding="utf-8"
)

print("app.py updated successfully.")
print(app_path)

ValueError: substring not found

In [ ]:
from pathlib import Path

app_path = Path(
    "/content/hospital_readmission_space/app.py"
)

app_text = app_path.read_text(
    encoding="utf-8"
)

# Hide the extra Gradio label on the Tab 1 plot
app_text = app_text.replace(
    """risk_plot = gr.Plot(
                    label=None
                )""",
    """risk_plot = gr.Plot(
                    show_label=False
                )"""
)

# Hide the extra Gradio label on the Tab 3 interpretation box
app_text = app_text.replace(
    """cost_interpretation_output = gr.Textbox(
                    label=None,
                    lines=6
                )""",
    """cost_interpretation_output = gr.Textbox(
                    show_label=False,
                    lines=6
                )"""
)

# Hide the extra Gradio label on the Tab 3 plot
app_text = app_text.replace(
    """cost_plot_output = gr.Plot(
                    label=None
                )""",
    """cost_plot_output = gr.Plot(
                    show_label=False
                )"""
)

app_path.write_text(
    app_text,
    encoding="utf-8"
)

print("Extra Textbox/Plot labels removed.")

Extra Textbox/Plot labels removed.


In [ ]:
!pip install -q huggingface_hub

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found in Colab Secrets."
    )

hf_api = HfApi(
    token=HF_TOKEN
)

print("Connected to Hugging Face.")

Connected to Hugging Face.


In [ ]:
HF_USERNAME = "chloeprowse"

SPACE_NAME = "hospital-readmission-decision-support"

SPACE_REPO = (
    f"{HF_USERNAME}/{SPACE_NAME}"
)

print(SPACE_REPO)

chloeprowse/hospital-readmission-decision-support


In [ ]:
from pathlib import Path
import shutil

SPACE_DIR = Path(
    "/content/hospital_readmission_space"
)

SPACE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Deployment folder created:")
print(SPACE_DIR)

Deployment folder created:
/content/hospital_readmission_space


In [ ]:
deployment_files = [
    "app (2).py",
    "requirements.txt",
    "README (1).md",
    "readmission_xgboost_model (2).pkl",
    "readmission_threshold (2).pkl",
    "rag_document_chunks.csv",
]

for file_name in deployment_files:

    source = Path(
        "/content"
    ) / file_name

    destination = (
        SPACE_DIR / file_name
    )

    if not source.exists():
        raise FileNotFoundError(
            f"Missing file: {source}"
        )

    shutil.copy(
        source,
        destination
    )

print("Deployment files copied.")

Deployment files copied.


In [ ]:
from pathlib import Path

SPACE_DIR = Path(
    "/content/hospital_readmission_space"
)

rename_map = {
    "app (2).py": "app.py",
    "README (1).md": "README.md",
    "readmission_threshold (2).pkl": "readmission_threshold.pkl",
    "readmission_xgboost_model (2).pkl": "readmission_xgboost_model.pkl",
}

for old_name, new_name in rename_map.items():

    old_path = SPACE_DIR / old_name
    new_path = SPACE_DIR / new_name

    if old_path.exists():
        old_path.rename(new_path)

        print(
            old_name,
            "->",
            new_name
        )

app (2).py -> app.py
README (1).md -> README.md
readmission_threshold (2).pkl -> readmission_threshold.pkl
readmission_xgboost_model (2).pkl -> readmission_xgboost_model.pkl


In [ ]:
hf_api.create_repo(
    repo_id=SPACE_REPO,
    repo_type="space",
    space_sdk="gradio",
    private=False,
    exist_ok=True
)

print("Space created.")

Space created.


In [ ]:
OPENAI_API_KEY = userdata.get(
    "OPENAI_API_KEY"
)

if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY was not found in Colab Secrets."
    )

In [ ]:
hf_api.add_space_secret(
    repo_id=SPACE_REPO,
    key="OPENAI_API_KEY",
    value=OPENAI_API_KEY
)

print(
    "OpenAI API key added as a Space secret."
)

OpenAI API key added as a Space secret.


In [ ]:
from pathlib import Path

readme_path = SPACE_DIR / "README.md"

readme_text = readme_path.read_text(
    encoding="utf-8"
)

readme_text = readme_text.replace(
    "colorTo: cyan",
    "colorTo: indigo"
)

readme_path.write_text(
    readme_text,
    encoding="utf-8"
)

print("README fixed.")

README fixed.


In [ ]:
print(
    readme_path.read_text(
        encoding="utf-8"
    )[:300]
)

---
title: Hospital Readmission Decision Support
emoji: 🏥
colorFrom: blue
colorTo: indigo
sdk: gradio
app_file: app.py
pinned: false
---

# Hospital Readmission Decision Support

This Gradio app estimates 30-day readmission risk using a saved XGBoost pipeline, provides evidence-based discharge suppo


In [ ]:
import sklearn
import xgboost
import joblib

print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("joblib:", joblib.__version__)

scikit-learn: 1.6.1
xgboost: 3.4.1
joblib: 1.5.3


In [ ]:
requirements_path = SPACE_DIR / "requirements.txt"

requirements_path.write_text(
    """gradio>=5.0
joblib==1.5.3
openai>=1.0
scikit-learn==1.6.1
xgboost==3.4.1
pandas>=2.0
numpy>=1.26
matplotlib>=3.8
""",
    encoding="utf-8"
)

print(
    requirements_path.read_text()
)

gradio>=5.0
joblib==1.5.3
openai>=1.0
scikit-learn==1.6.1
xgboost==3.4.1
pandas>=2.0
numpy>=1.26
matplotlib>=3.8



In [ ]:
from pathlib import Path

SPACE_DIR = Path(
    "/content/hospital_readmission_space"
)

app_path = SPACE_DIR / "app.py"

print("Updating:")
print(app_path)

Updating:
/content/hospital_readmission_space/app.py


In [ ]:
hf_api.upload_folder(
    repo_id=SPACE_REPO,
    repo_type="space",
    folder_path=str(
        SPACE_DIR
    ),
    commit_message=(
        "Fix application filenames "
        "and deploy dashboard"
    )
)

print("Dashboard uploaded successfully.")

Dashboard uploaded successfully.


In [ ]:
SPACE_URL = (
    f"https://huggingface.co/spaces/"
    f"{SPACE_REPO}"
)

print("Hugging Face Space:")
print(SPACE_URL)

Hugging Face Space:
https://huggingface.co/spaces/chloeprowse/hospital-readmission-decision-support
